# *This notebook provides a workflow to carry out a benchmark on the annotation-based databases considered. Two hyperparameters are tested*:
- the damping factor *d*
- (for STRING only) the confidence score *CS* threshold used to filter out the data

A report is then written to list the best hyperparameters for each database and to mention the best database for the biological process of interest.

### *Importing the required libraries*

In [1]:
import glob
import networkx as nx
import os
import pandas as pd
import re
import shutil
import sys
sys.path.append("../scripts")

import useful_functions 
from tqdm import tqdm

### *Reading the training genes and setting the input parameters*

In [ ]:
# Reading the training genes
genes = pd.read_csv("../TrainingGenes/training_genes_stalk_cell.csv")
genes = genes["Feature"].to_list()

# Storing all the PPI databases in a list
databases = ["BioGrid", "ConsensusPath", "GeneCards", "STRING"]

# Setting the process of interest
process = "stalk_cell"

# Setting the path of the hyperparameters folder
path_hyperparameters = "../graphs/databases/hyperparameters"

if not os.path.exists(path_hyperparameters):
    os.mkdir(path_hyperparameters)

### *Initializing the loop*

In [ ]:
# Initiating the loop
for database in tqdm(databases, desc = "Processing benchmarking on PPI databases ..."):

    # Setting the path for the hyperparameters files that will be generated
    path_data = f"{path_data}/{database}"

    if not os.path.exists(path_data):
        os.mkdir(path_data)

    # Create a folder for the upcoming threshold benchmark
    path_benchmark = f"../results/{process}"
    path_benchmark_database = f"{path_benchmark}/{database}"
    path_results = f"{path_benchmark_database}/Benchmark"

    if not os.path.exists(f"{path_benchmark}"):
        os.mkdir(f"{path_benchmark}")

    if not os.path.exists(f"{path_benchmark_database}"):
        os.mkdir(f"{path_benchmark_database}")

    if not os.path.exists(f"{path_results}"):
        os.mkdir(f"{path_results}")
    else:
        shutil.rmtree(f"{path_results}")
        os.mkdir(f"{path_results}")
   
    # Reading the graph file
    G = nx.read_graphml(f"{path_data}/graph_{database}.graphml")

    # Running a benchmark to determine the best damping factor
    benchmark_results = useful_functions.analyse(G, genes)
    benchmark_results.to_csv(f"{path_results}/PageRank_evaluation_{database}_{process}.csv",
                             sep = ",", index = False)

    # Retrieving the best DF and saving the file as hyperparameters
    hyperparameters_df = benchmark_results[benchmark_results["auc_roc"] == benchmark_results["auc_roc"].max()]
    hyperparameters_df.to_csv(f"{path_data}/Hyperparameters_{database}_{process}.csv", sep = ",", index = False)

### *Analyzing the benchmark results*

In [ ]:
# Analyzing the benchmark results to determine which PPI database is the best for the specific process considered
# Step 1: Cleaning the files
files = glob.glob(f"../graphs/databases/hyperparameters/*/Hyperparameters_*{process}.csv")
for file in files:
    
    # Extracting the dataset name
    filename = os.path.basename(file)
    database = filename.split("_")[1]

    # The workflow differs for the STRING database
    if database == "STRING":
        df = pd.read_csv(file)
        df["Database"] = database

        new_cols = ["Database", "Threshold", "Best_AUC", "Best_DF"]
        df = df.reindex(columns = new_cols)
        df.to_csv(file, sep = ",", index = False)

    else:
        df = pd.read_csv(file)
        df["Database"] = database
        df["Threshold"] = "None"

        # Extracting the columns of interest
        df = df[["Database", "Threshold", "auc_roc", "damping_factor"]]

        # Renaming the columns
        df = df.rename(columns = {"auc_roc": "Best_AUC", "damping_factor": "Best_DF"})

        # Saving the clean file
        df.to_csv(file, sep = ",", index = False)
 
# Step 2: Concatenating the files into one and retrieving the best dataset for the process considered
files = glob.glob(f"../graphs/databases/hyperparameters/Hyperparameters_*{process}*.csv")

df_best = pd.concat(map(pd.read_csv, files), ignore_index = True)
df_best = df_best[df_best["Best_AUC"] == df_best["Best_AUC"].max()]

# Step 3: Writting a .txt storing the best omics dataset parameters
if df_best["Database"].values[0] == "STRING": 
    with open(f"../graphs/databases/Best_PPI_database_{process}.txt", "w") as f_out:
        f_out.write(f"Best PPI database for {process} process: {df_best['Database'].values[0]}\n")
        f_out.write(f"---> CS threshold: {df_best['Threshold'].values[0]}\n")
        f_out.write(f"---> damping factor: {df_best['Best_DF'].values[0]}\n")

    print(f"Best PPI database for {process} process: {df_best['Database'].values[0]}")
    print(f"---> CS threshold: {df_best['Threshold'].values[0]}")
    print(f"---> damping factor: {df_best['Best_DF'].values[0]}")

else:
    with open(f"../graphs/databases/Best_PPI_database_{process}.txt", "w") as f_out:
        f_out.write(f"Best PPI database for {process} process: {df_best['Database'].values[0]}\n")
        f_out.write(f"---> damping factor: {df_best['Best_DF'].values[0]}\n")

    print(f"Best PPI database for {process} process: {df_best['Database'].values[0]}")
    print(f"---> damping factor: {df_best['Best_DF'].values[0]}")